In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Scaled dot-product attention bằng NumPy

Trừ max trước softmax để ổn định số; mask phải broadcast đúng về `[Lq,Lk]`.

In [ ]:
def softmax(x,axis=-1):
    z=x-np.max(x,axis=axis,keepdims=True); e=np.exp(z); return e/e.sum(axis=axis,keepdims=True)
def attention(q,k,v,mask=None):
    """q[...,Lq,D], k/v[...,Lk,D/Dv]."""
    scores=q@np.swapaxes(k,-1,-2)/math.sqrt(q.shape[-1])
    if mask is not None: scores=np.where(mask,scores,-1e30)
    weights=softmax(scores,-1); return weights@v,weights
rng=np.random.default_rng(42); q=rng.normal(size=(2,3,4)); k=rng.normal(size=(2,5,4)); v=rng.normal(size=(2,5,6))
out,w=attention(q,k,v); assert out.shape==(2,3,6); assert np.allclose(w.sum(-1),1)
mask=np.tril(np.ones((3,3),dtype=bool)); o,m=attention(q[:,:3],q[:,:3],q[:,:3],mask)
assert np.allclose(m*np.triu(np.ones((3,3)),1),0); print(out.shape,w[0,0])